In [ ]:
import os
import numpy as np
import pandas as pd
from PIL import Image

from tensorflow.keras.applications.vgg16 import VGG16, preprocess_input
from tensorflow.keras.preprocessing.image import img_to_array
from tensorflow.keras.models import Model

In [9]:

base_model = VGG16(weights="imagenet")

# Use FC layer (4096 features)
model = Model(
    inputs=base_model.input,
    outputs=base_model.get_layer("fc2").output
)


In [12]:
IMAGE_SIZE = (224, 224)   # ✅ MUST match your preprocessing
DATASET_PATH = "voxceleb/mel/"

X = []
y = []

batch_images = []
batch_labels = []

BATCH_SIZE = 32


In [13]:
for speaker in os.listdir(DATASET_PATH):

    speaker_path = os.path.join(DATASET_PATH, speaker)

    if not os.path.isdir(speaker_path):
        continue

    for file in os.listdir(speaker_path):

        if not file.lower().endswith(".jpg"):
            continue

        img_path = os.path.join(speaker_path, file)

        try:
            with Image.open(img_path) as img:
                img = img.convert("RGB")
                img = img.resize(IMAGE_SIZE)
                img = img_to_array(img)

            batch_images.append(img)
            batch_labels.append(speaker)

            # Process batch
            if len(batch_images) == BATCH_SIZE:
                batch_np = preprocess_input(np.array(batch_images))
                features = model.predict(batch_np, verbose=0)

                for f, label in zip(features, batch_labels):
                    X.append(f)
                    y.append(label)

                batch_images, batch_labels = [], []

        except Exception as e:
            print(f"Error: {img_path} -> {e}")


In [14]:
if len(batch_images) > 0:
    batch_np = preprocess_input(np.array(batch_images))
    features = model.predict(batch_np, verbose=0)

    for f, label in zip(features, batch_labels):
        X.append(f)
        y.append(label)


In [15]:
X = np.array(X)
y = np.array(y)

print("Feature shape:", X.shape)   # (num_samples, 4096)
print("Labels:", len(y))

# Create DataFrame
df = pd.DataFrame(X)
df["label"] = y

Feature shape: (12105, 4096)
Labels: 12105


In [ ]:
csv_path = "dataset/voxceleb/voxceleb_features.csv"

# Ensure parent directory exists
os.makedirs(os.path.dirname(csv_path), exist_ok=True)

# Save CSV
df.to_csv(csv_path, index=False)
print("✅ CSV saved at:", csv_path)


✅ CSV saved at: dataset/voxceleb/vgg16_voxceleb_features.csv
